# Third-Party Vendor Governance Demo## CloudFirst Financial: Vendor Risk Scoring (2 Vendors)Quick demo showing vendor risk scoring and SLA compliance monitoring for CloudFirst Financial's fraud detection AI system.---### Quick Reference**Risk Score Scale**: 1 (Lowest Risk) → 5 (Highest Risk)**Scoring Weights**: Security (30%), Data Handling (25%), Transparency (20%), Financial Stability (15%), SLA Quality (10%)**SLA Thresholds**: Uptime ≥ 99.9%, Latency ≤ 800ms### ScenarioCloudFirst Financial evaluates 2 AI vendors (CloudVault AI and TitanScale AI) for their fraud detection system after a competitor suffered a 6-hour vendor outage.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("CloudFirst Financial Vendor Governance Demo")
print("="*50)

## Step 1: Define Vendor Data

In [2]:
# Vendor evaluation data for CloudVault AI and TitanScale AIvendors_df = pd.DataFrame({    'Vendor': ['CloudVault AI', 'TitanScale AI'],    'Security_Score': [95, 95],    'Data_Handling_Score': [90, 85],    'Model_Transparency': [70, 70],    'Financial_Stability': [100, 100],    'SLA_Compliance': [90, 95],    'Exit_Strategy': [95, 90]})print("\nVendor Scores:")print(vendors_df)

## Step 2: Calculate Risk Scores

In [3]:
def calculate_vendor_risk_score(vendor_data):
    """Calculate weighted vendor risk score (1-5 scale)"""
    weights = {
        'Security_Score': 0.30,
        'Data_Handling_Score': 0.25,
        'Model_Transparency': 0.20,
        'Financial_Stability': 0.15,
        'SLA_Compliance': 0.10
    }
    
    risk_scores = []
    for idx, row in vendor_data.iterrows():
        weighted_score = sum(
            row[metric] * weight 
            for metric, weight in weights.items()
        )
        # Convert to risk score (1-5)
        risk_level = 5 - (weighted_score / 100) * 4
        risk_scores.append(round(risk_level, 2))
    
    return risk_scores

vendors_df['Risk_Score'] = calculate_vendor_risk_score(vendors_df)

print("\nRisk Scores Calculated:")
print(vendors_df[['Vendor', 'Risk_Score']])
print("\nScale: 1 (Lowest Risk) to 5 (Highest Risk)")

## Step 3: Generate SLA Data

In [4]:
# Generate 30 days of SLA datanp.random.seed(42)days = 30sla_data = []base_uptime = {'CloudVault AI': 0.9995, 'TitanScale AI': 0.9999}base_latency = {'CloudVault AI': 400, 'TitanScale AI': 300}for vendor in vendors_df['Vendor']:    for day in range(1, days + 1):        date = (datetime.now() - timedelta(days=days-day)).date()        uptime = base_uptime[vendor] + np.random.normal(0, 0.0002)        uptime = min(1.0, max(0.98, uptime))        latency = base_latency[vendor] + np.random.normal(0, 50)        latency = max(200, latency)                sla_data.append({            'Date': date,            'Vendor': vendor,            'Uptime': round(uptime * 100, 3),            'Latency_ms': round(latency, 1),            'Day': day        })sla_df = pd.DataFrame(sla_data)print(f"SLA data generated: {len(sla_df)} records")print(sla_df.head(10))

## Step 4: Analyze Compliance

In [5]:
# Calculate compliance metrics
uptime_threshold = 99.9
latency_threshold = 800

compliance_df = sla_df.groupby('Vendor').agg({
    'Uptime': ['mean', 'min', 'max'],
    'Latency_ms': ['mean', 'min', 'max']
}).round(2)

compliance_df.columns = ['_'.join(col).strip() for col in compliance_df.columns.values]
compliance_df = compliance_df.reset_index()

compliance_df['Uptime_Compliant'] = compliance_df['Uptime_mean'] >= uptime_threshold
compliance_df['Latency_Compliant'] = compliance_df['Latency_ms_mean'] <= latency_threshold

print("\nSLA Compliance Summary:")
print(compliance_df)

## Step 5: Comparison Chart

In [6]:
# Create risk comparison chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Risk score comparison
sorted_vendors = vendors_df.sort_values('Risk_Score')
colors = ['#2ecc71' if score <= 1.5 else '#f39c12' for score in sorted_vendors['Risk_Score']]
ax1.barh(sorted_vendors['Vendor'], sorted_vendors['Risk_Score'], color=colors)
ax1.set_xlabel('Risk Score (1 = Lowest, 5 = Highest)', fontweight='bold')
ax1.set_title('Vendor Risk Comparison', fontweight='bold')
ax1.set_xlim(0, 5)
ax1.axvline(x=2, color='gray', linestyle='--', alpha=0.5)

# SLA uptime comparison
uptime_means = compliance_df.sort_values('Uptime_mean', ascending=True)
ax2.barh(uptime_means['Vendor'], uptime_means['Uptime_mean'], color='#3498db')
ax2.axvline(x=uptime_threshold, color='red', linestyle='--', linewidth=2, label=f'SLA Threshold ({uptime_threshold}%)')
ax2.set_xlabel('Average Uptime (%)', fontweight='bold')
ax2.set_title('30-Day Uptime Comparison', fontweight='bold')
ax2.legend()
ax2.set_xlim(99.8, 100.1)

plt.tight_layout()
plt.savefig('vendor_comparison_demo.png', dpi=150, bbox_inches='tight')
plt.show()

print("Chart saved as: vendor_comparison_demo.png")

## Step 6: Recommendations

In [7]:
# Generate recommendations
print("\n" + "="*70)
print("VENDOR ASSESSMENT & RECOMMENDATIONS")
print("="*70)

for idx, vendor in enumerate(vendors_df['Vendor']):
    vendor_row = vendors_df[vendors_df['Vendor'] == vendor].iloc[0]
    compliance_row = compliance_df[compliance_df['Vendor'] == vendor].iloc[0]
    
    risk_score = vendor_row['Risk_Score']
    uptime = compliance_row['Uptime_mean']
    latency = compliance_row['Latency_ms_mean']
    
    # Determine recommendation
    if risk_score <= 1.5 and uptime >= uptime_threshold:
        recommendation = "Highly Recommended"
    elif risk_score <= 2.5 and uptime >= uptime_threshold:
        recommendation = "Recommended"
    else:
        recommendation = "Acceptable (with monitoring)"
    
    print(f"\n{idx+1}. {vendor}")
    print(f"   Risk Score: {risk_score:.2f}/5.0")
    print(f"   Avg Uptime: {uptime:.3f}% (SLA: {uptime_threshold}%) {'✓' if uptime >= uptime_threshold else '✗'}")
    print(f"   Avg Latency: {latency:.1f}ms (Target: <{latency_threshold}ms) {'✓' if latency <= latency_threshold else '✗'}")
    print(f"   Recommendation: {recommendation}")

print("\n" + "="*70)